In [7]:
import numpy as np
import pandas as pd
import shutil
from pathlib import Path

# 1. 設定路徑
data_dir = Path("D:/hagrid_project/dataset_v2_processed")
csv_path = data_dir / "labels.csv"
need_check_dir = data_dir / "need_check"

# 建立 need_check 資料夾
if need_check_dir.exists():
    shutil.rmtree(need_check_dir) # 如果之前有跑過，先清空
need_check_dir.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(csv_path)

# --- 關鍵修正：路徑自動修正 ---
def fix_path(path_str):
    filename = Path(path_str).name
    if "landmark" in str(path_str):
        return str(data_dir / "landmarks" / filename)
    else:
        return str(data_dir / "crops" / filename)

df['landmark_path'] = df['landmark_path'].apply(fix_path)
df['crop_path'] = df['crop_path'].apply(fix_path)

def dist(a, b):
    return np.linalg.norm(a - b)

def get_finger_states(lm):
    wrist = lm[0]
    palm_scale = dist(lm[0], lm[9]) + 1e-6
    # 寬鬆版：threshold 設為 0.0
    def is_extended(mcp, pip, dip, tip, threshold=0.0):
        return dist(lm[tip], wrist) > dist(lm[pip], wrist) + threshold * palm_scale

    return {
        "thumb": dist(lm[4], (lm[0]+lm[5]+lm[9]+lm[13]+lm[17])/5) > 0.45 * palm_scale,
        "index": is_extended(5, 6, 7, 8),
        "middle": is_extended(9, 10, 11, 12),
        "ring": is_extended(13, 14, 15, 16),
        "pinky": is_extended(17, 18, 19, 20),
    }

def check_expected(label_name, lm):
    states = get_finger_states(lm)
    thumb, index = states["thumb"], states["index"]
    middle, ring, pinky = states["middle"], states["ring"], states["pinky"]
    extended_count = sum(states.values())
    palm_scale = dist(lm[0], lm[9]) + 1e-6
    thumb_index_tip_dist = dist(lm[4], lm[8]) / palm_scale

    # if label_name == "fist":
    #     # fist: 大部分手指應該是彎的
    #     ok = extended_count <= 1

    # elif label_name == "like":
    #     # like: 拇指伸出，其他四指大多彎曲
    #     ok = thumb and (not index) and (not middle) and (not ring) and (not pinky)

    # elif label_name == "one":
    #     # one: 食指伸出，其他大多彎曲
    #     ok = index and (not middle) and (not ring) and (not pinky)

    # elif label_name == "palm":
    #     # palm: 至少四根手指伸出
    #     ok = extended_count >= 4

    # elif label_name == "ok":
    #     # ok: 拇指尖和食指尖靠近
    #     # middle/ring/pinky 通常會伸出，但先不要設太嚴
    #     ok = thumb_index_tip_dist < 0.45

    # else:
    #     # N/A 不檢查 target pattern
    #     ok = True


    if label_name == "fist":
        ok = extended_count <= 2 # 允許微鬆的拳頭
    elif label_name == "like":
        ok = thumb and (not index) and (not middle) # 只看拇指、食指、中指
    elif label_name == "one":
        ok = index and (not middle) and (not ring)
    elif label_name == "palm":
        ok = extended_count >= 3 # 至少3根手指
    elif label_name == "ok":
        ok = thumb_index_tip_dist < 0.8 # 巨幅放寬 OK 的距離
    else:
        ok = True

    return ok

suspicious_ids = []

print("開始掃描可疑圖片...")
for row in df.itertuples():
    if row.label_name == "N_A":
        continue
        
    lm = np.load(row.landmark_path)
    if not check_expected(row.label_name, lm):
        suspicious_ids.append(row.idx)
        
        # 依照標籤建立子資料夾，方便人類看
        label_dir = need_check_dir / row.label_name
        label_dir.mkdir(exist_ok=True)
        
        # 將可疑圖片複製過去，檔名加上 idx_ 原檔名
        dest_path = label_dir / f"{row.idx}_{Path(row.crop_path).name}"
        shutil.copy(row.crop_path, dest_path)

print(f"掃描完成！共有 {len(df)} 張目標手勢。")
print(f"抓出 {len(suspicious_ids)} 張可疑圖片，已複製到 {need_check_dir} 中。")

開始掃描可疑圖片...
掃描完成！共有 99996 張目標手勢。
抓出 3271 張可疑圖片，已複製到 D:\hagrid_project\dataset_v2_processed\need_check 中。


In [8]:
import pandas as pd
from pathlib import Path

# 1. 設定你的 CSV 路徑
csv_path = Path("D:/hagrid_project/dataset_v2_processed/labels.csv")
output_path = Path("D:/hagrid_project/dataset_v2_processed/labels_fixed.csv")

# 2. 讀取資料表
df = pd.read_csv(csv_path)

# 3. 執行「一鍵轉 N/A」
# 假設 suspicious_ids 是你剛才透過規則抓出來的 324 個嫌疑犯索引清單
# 如果你剛好沒有那個清單變數，也可以直接用過濾器重新篩選一次
# 這裡我們將所有 label 改為 0，label_name 改為 N_A
df.loc[df['idx'].isin(suspicious_ids), 'label'] = 0
df.loc[df['idx'].isin(suspicious_ids), 'label_name'] = 'N_A'

# 4. 存檔
df.to_csv(output_path, index=False)

print(f"✅ 大功告成！")
print(f"已將 {len(suspicious_ids)} 筆疑似垃圾資料轉為 N_A (Class 0)")
print(f"新檔案已儲存至: {output_path}")

✅ 大功告成！
已將 3271 筆疑似垃圾資料轉為 N_A (Class 0)
新檔案已儲存至: D:\hagrid_project\dataset_v2_processed\labels_fixed.csv
